# Step 3: Model Selection, Training and Evaluation

**Dataset**: Cardiotocography (CTG) — Fetal Heart Rate Signal Classification  
**Task**: Multi-class classification → Predict fetal state (`NSP`)  
- 1 = Normal  
- 2 = Suspect  
- 3 = Pathologic  

**Notebook Sections**:
1. Load Data & Import Libraries
2. Data Preprocessing, EDA & Train-Test Split
3. Model Training (Base Models)
4. Model Evaluation
5. Testing on New Instances
6. Hyperparameter Tuning
7. Model Serialization

---
## 3.1 Load Data and Import Libraries

In [ ]:
# ──────────────────────────────────────────────────────────────
# Standard Libraries
# ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import os

# ──────────────────────────────────────────────────────────────
# Scikit-learn: Preprocessing
# ──────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ──────────────────────────────────────────────────────────────
# Scikit-learn: Models
# ──────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# ──────────────────────────────────────────────────────────────
# Scikit-learn: Evaluation
# ──────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# ──────────────────────────────────────────────────────────────
# Model Serialization
# ──────────────────────────────────────────────────────────────
import joblib

# ──────────────────────────────────────────────────────────────
# VIF for Multicollinearity
# ──────────────────────────────────────────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ──────────────────────────────────────────────────────────────
# Settings
# ──────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All libraries imported successfully.")

### Load the Dataset

In [ ]:
# Load the CTG dataset from CSV
df = pd.read_csv("CTG_data.csv")

print(f"Original shape: {df.shape}")
print(f"\nColumn names ({len(df.columns)}):")
print(list(df.columns))
df.head()

In [ ]:
# Basic information about the dataset
df.info()

---
## 3.2 Data Preprocessing, Exploratory Data Analysis (EDA), and Train-Test Split

### 3.2.1 Data Preprocessing

In [ ]:
# ──────────────────────────────────────────────────────────────
# Step 1: Drop rows that are entirely NaN (the first row in
#         the raw sheet is blank metadata)
# ──────────────────────────────────────────────────────────────
df = df.dropna(how="all")
print(f"Shape after dropping all-NaN rows: {df.shape}")

# ──────────────────────────────────────────────────────────────
# Step 2: Drop non-feature / metadata columns
#   - FileName, Date, SegFile : exam identifiers
#   - b, e                    : start/end instant (not clinical features)
#   - LBE                     : baseline by expert (redundant with LB)
# ──────────────────────────────────────────────────────────────
cols_to_drop_metadata = ["FileName", "Date", "SegFile", "b", "e", "LBE"]
df.drop(columns=cols_to_drop_metadata, inplace=True)
print(f"Shape after dropping metadata columns: {df.shape}")

# ──────────────────────────────────────────────────────────────
# Step 3: Drop individual morphologic pattern columns
#   - A, B, C, D, E, AD, DE, LD, FS, SUSP : binary class indicators
#   - CLASS : 10-class target (we use NSP instead)
#   - DR    : constant zero (p(Kruskal-Wallis) = 1, no variance)
# ──────────────────────────────────────────────────────────────
pattern_cols = ["A", "B", "C", "D", "E", "AD", "DE", "LD", "FS", "SUSP"]
df.drop(columns=pattern_cols, inplace=True)
df.drop(columns=["CLASS"], inplace=True)
df.drop(columns=["DR"], inplace=True)
print(f"Shape after dropping pattern/class/DR columns: {df.shape}")
print(f"\nRemaining columns: {list(df.columns)}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Step 4: Handle missing values
# ──────────────────────────────────────────────────────────────
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Drop any remaining rows with missing target
df = df.dropna(subset=["NSP"])

# Impute remaining missing values with median (robust to outliers)
if df.isnull().sum().sum() > 0:
    print(f"\nImputing {df.isnull().sum().sum()} missing values with column medians...")
    df = df.fillna(df.median(numeric_only=True))

print(f"\nFinal shape after handling missing values: {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Step 5: Check for duplicate rows
# ──────────────────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Shape after removing duplicates: {df.shape}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Step 6: Convert target to integer type
# ──────────────────────────────────────────────────────────────
df["NSP"] = df["NSP"].astype(int)

print("Preprocessed dataset preview:")
df.head()

### 3.2.2 Exploratory Data Analysis (EDA)

In [ ]:
# ──────────────────────────────────────────────────────────────
# Summary Statistics
# ──────────────────────────────────────────────────────────────
df.describe().T

In [ ]:
# ──────────────────────────────────────────────────────────────
# Target Class Distribution
# ──────────────────────────────────────────────────────────────
nsp_labels = {1: "Normal", 2: "Suspect", 3: "Pathologic"}
nsp_counts = df["NSP"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = axes[0].bar(
    [nsp_labels[k] for k in nsp_counts.index],
    nsp_counts.values,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
)
for bar, count in zip(bars, nsp_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
                 str(count), ha="center", fontweight="bold", fontsize=12)
axes[0].set_title("NSP Class Distribution", fontweight="bold", fontsize=14)
axes[0].set_ylabel("Count")

# Pie chart
axes[1].pie(
    nsp_counts.values,
    labels=[f"{nsp_labels[k]}\n({v})" for k, v in zip(nsp_counts.index, nsp_counts.values)],
    colors=colors,
    autopct="%1.1f%%",
    startangle=140,
    textprops={"fontsize": 11},
    wedgeprops={"edgecolor": "black", "linewidth": 0.5},
)
axes[1].set_title("NSP Class Proportions", fontweight="bold", fontsize=14)

plt.tight_layout()
plt.show()

print("⚠️ The dataset is IMBALANCED — Normal class dominates (~78%).")
print("   → We will use Stratified K-Fold CV and stratified train-test split.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Feature Distributions (Histograms)
# ──────────────────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != "NSP"]

n_plot_cols = 4
n_plot_rows = (len(feature_cols) + n_plot_cols - 1) // n_plot_cols  # ceiling division

fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(20, 4 * n_plot_rows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    for nsp_val, color, label in zip([1, 2, 3], colors, ["Normal", "Suspect", "Pathologic"]):
        subset = df[df["NSP"] == nsp_val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor="black", linewidth=0.3)
    ax.set_title(col, fontweight="bold", fontsize=11)
    ax.tick_params(labelsize=9)

# Remove unused subplots
for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

axes[0].legend(fontsize=9)
fig.suptitle("Feature Distributions by NSP Class", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────
# Correlation Heatmap
# ──────────────────────────────────────────────────────────────
# Only include numeric columns (exclude NSP target for feature correlations)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

# Dynamically size the figure based on feature count
n_feats = len(corr_matrix.columns)
fig_size = max(10, n_feats * 0.7)
annot_size = max(6, min(9, 180 // n_feats))

plt.figure(figsize=(fig_size, fig_size * 0.85))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": annot_size},
)
plt.title("Feature Correlation Heatmap", fontsize=16, fontweight="bold", pad=20)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print highly correlated pairs (|r| > 0.7)
print("\nHighly correlated feature pairs (|r| > 0.7):")
print("-" * 50)
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            print(f"  {corr_matrix.columns[i]:>10} <-> {corr_matrix.columns[j]:<10} : r = {r:.3f}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Box Plots for Outlier Detection
# ──────────────────────────────────────────────────────────────
n_plot_cols = 4
n_plot_rows = (len(feature_cols) + n_plot_cols - 1) // n_plot_cols

fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(20, 4 * n_plot_rows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(data=df, x="NSP", y=col, ax=axes[i], palette=colors,
                hue="NSP", legend=False)
    axes[i].set_title(col, fontweight="bold", fontsize=11)
    axes[i].set_xlabel("")
    axes[i].set_xticklabels(["Normal", "Suspect", "Pathologic"], fontsize=9)

for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Box Plots - Outlier Detection by NSP Class", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────
# Outlier Handling — IQR Capping (Winsorization)
# ──────────────────────────────────────────────────────────────
print("Outlier capping using IQR method (1.5 × IQR):")
print("=" * 55)

for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers_below = (df[col] < lower).sum()
    outliers_above = (df[col] > upper).sum()
    total_outliers = outliers_below + outliers_above

    if total_outliers > 0:
        print(f"  {col:>10}: {total_outliers:3d} outliers capped  "
              f"(below={outliers_below}, above={outliers_above})")
        df[col] = df[col].clip(lower=lower, upper=upper)

print("\n✅ Outlier capping complete.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Multicollinearity — Variance Inflation Factor (VIF)
# ──────────────────────────────────────────────────────────────
X_vif = df[feature_cols].copy()
X_vif = X_vif.assign(const=1)  # add intercept for VIF calculation

vif_data = pd.DataFrame({
    "Feature": feature_cols,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(len(feature_cols))
    ],
})
vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)

print("Variance Inflation Factor (VIF) Analysis")
print("=" * 45)
print(vif_data.to_string(index=False))
print("\nRule of thumb: VIF > 10 → high multicollinearity")

# Flag features with high VIF
high_vif = vif_data[vif_data["VIF"] > 10]
if len(high_vif) > 0:
    print(f"\n⚠️ Features with VIF > 10: {list(high_vif['Feature'])}")
    print("   These features show high multicollinearity.")
    print("   Dropping features iteratively to reduce VIF...")

    # Iteratively drop the feature with the highest VIF until all VIF < 10
    dropped_features = []
    current_features = feature_cols.copy()

    while True:
        X_temp = df[current_features].copy()
        X_temp = X_temp.assign(const=1)
        vifs = [variance_inflation_factor(X_temp.values, i) for i in range(len(current_features))]
        max_vif_idx = np.argmax(vifs)
        max_vif = vifs[max_vif_idx]

        if max_vif <= 10:
            break

        drop_col = current_features[max_vif_idx]
        dropped_features.append(drop_col)
        current_features.remove(drop_col)
        print(f"   Dropped: {drop_col} (VIF = {max_vif:.1f})")

    feature_cols = current_features
    print(f"\n✅ Remaining features ({len(feature_cols)}): {feature_cols}")
else:
    print("\n✅ No features with VIF > 10. No multicollinearity issues.")

### 3.2.3 Train-Test Split

In [ ]:
# ──────────────────────────────────────────────────────────────
# Separate Features and Target
# ──────────────────────────────────────────────────────────────
X = df[feature_cols].copy()
y = df["NSP"].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts().sort_index().to_frame("count"))

In [ ]:
# ──────────────────────────────────────────────────────────────
# Stratified 80/20 Train-Test Split
# ──────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,  # preserve class proportions
)

print(f"Training set:  X={X_train.shape}, y={y_train.shape}")
print(f"Testing set:   X={X_test.shape}, y={y_test.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts().sort_index().to_frame("count"))
print(f"\nTesting class distribution:")
print(y_test.value_counts().sort_index().to_frame("count"))

In [ ]:
# ──────────────────────────────────────────────────────────────
# Feature Scaling with StandardScaler
# ──────────────────────────────────────────────────────────────
scaler = StandardScaler()

# Fit on training data ONLY, then transform both sets
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

print("✅ Feature scaling applied (StandardScaler).")
print(f"\nScaled training data sample:")
X_train_scaled.head()

---
## 3.3 Model Training

We train **3 base models** with default hyperparameters:

| # | Model | Rationale |
|---|---|---|
| 1 | **Logistic Regression** | Linear baseline, fast, interpretable |
| 2 | **Random Forest** | Ensemble of decision trees, handles non-linearity |
| 3 | **K-Nearest Neighbors (KNN)** | Instance-based learning, captures local patterns |

In [ ]:
# ──────────────────────────────────────────────────────────────
# Initialize Base Models
# ──────────────────────────────────────────────────────────────
base_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
    ),
    "KNN": KNeighborsClassifier(),
}

print("Base models initialized:")
for name, model in base_models.items():
    print(f"  • {name}: {model}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Train All Base Models
# ──────────────────────────────────────────────────────────────
trained_models = {}

for name, model in base_models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print("✅ Done")

print(f"\n✅ All {len(trained_models)} base models trained successfully.")

---
## 3.4 Model Evaluation

### 3.4.1 Classification Reports & Confusion Matrices

In [ ]:
# ──────────────────────────────────────────────────────────────
# Evaluate Each Model — Classification Report + Confusion Matrix
# ──────────────────────────────────────────────────────────────
all_reports = {}
target_names = ["Normal", "Suspect", "Pathologic"]

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)

    # Classification report
    report_dict = classification_report(
        y_test, y_pred,
        target_names=target_names,
        output_dict=True,
    )
    all_reports[name] = report_dict

    # Print report
    print("=" * 65)
    print(f"  {name}")
    print("=" * 65)
    print(classification_report(y_test, y_pred, target_names=target_names))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=target_names,
        yticklabels=target_names,
        linewidths=1,
        linecolor="black",
        cbar_kws={"shrink": 0.8},
    )
    plt.title(f"Confusion Matrix — {name}", fontsize=14, fontweight="bold")
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("Actual", fontsize=12)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# ──────────────────────────────────────────────────────────────
# Save Classification Reports to JSON
# ──────────────────────────────────────────────────────────────
with open("classification_report.json", "w") as f:
    json.dump(all_reports, f, indent=4)

print("✅ Classification reports saved to: classification_report.json")
print(f"   Models included: {list(all_reports.keys())}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Model Comparison Summary
# ──────────────────────────────────────────────────────────────
summary_data = []
for name, report in all_reports.items():
    summary_data.append({
        "Model": name,
        "Accuracy": report["accuracy"],
        "Macro Precision": report["macro avg"]["precision"],
        "Macro Recall": report["macro avg"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"],
        "Weighted F1": report["weighted avg"]["f1-score"],
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Weighted F1", ascending=False).reset_index(drop=True)

print("Model Comparison (Base Models):")
print("=" * 80)
print(summary_df.to_string(index=False, float_format="{:.4f}".format))

# Highlight the best model
best_base = summary_df.iloc[0]["Model"]
print(f"\n🏆 Best base model: {best_base}")

### 3.4.2 Cross-Validation

In [ ]:
# ──────────────────────────────────────────────────────────────
# K-Fold Cross-Validation (K=10)
# ──────────────────────────────────────────────────────────────
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print("K-Fold Cross-Validation (K=10)")
print("=" * 55)

kfold_results = {}
for name, model in base_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf, scoring="accuracy")
    kfold_results[name] = scores
    print(f"  {name:>22}: Mean = {scores.mean():.4f} ± {scores.std():.4f}  "
          f"(Scores: {np.round(scores, 3)})")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Stratified K-Fold Cross-Validation (K=10)
# ──────────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print("Stratified K-Fold Cross-Validation (K=10)")
print("=" * 55)
print("(Preserves class distribution in each fold — critical for imbalanced data)")
print()

skfold_results = {}
for name, model in base_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring="accuracy")
    skfold_results[name] = scores
    print(f"  {name:>22}: Mean = {scores.mean():.4f} ± {scores.std():.4f}  "
          f"(Scores: {np.round(scores, 3)})")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cross-Validation Comparison Visualization
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Fold results
kf_df = pd.DataFrame(kfold_results)
axes[0].boxplot(kf_df.values, labels=kf_df.columns, patch_artist=True,
                boxprops=dict(facecolor="#3498db", alpha=0.7),
                medianprops=dict(color="red", linewidth=2))
axes[0].set_title("K-Fold CV (K=10)", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_ylim(0.8, 1.0)

# Stratified K-Fold results
skf_df = pd.DataFrame(skfold_results)
axes[1].boxplot(skf_df.values, labels=skf_df.columns, patch_artist=True,
                boxprops=dict(facecolor="#2ecc71", alpha=0.7),
                medianprops=dict(color="red", linewidth=2))
axes[1].set_title("Stratified K-Fold CV (K=10)", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_ylim(0.8, 1.0)

plt.suptitle("Cross-Validation Results Comparison", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3.5 Testing on New Instances

Test each trained base model on new (unseen) data points to demonstrate inference.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Create New Test Instances
# Sample a few instances from the test set as "new" data points
# ──────────────────────────────────────────────────────────────
nsp_map = {1: "Normal", 2: "Suspect", 3: "Pathologic"}

# Pick one instance from each class for demonstration
new_instances = []
for cls in [1, 2, 3]:
    idx = y_test[y_test == cls].index[0]
    new_instances.append(idx)

X_new = X_test_scaled.loc[new_instances]
y_actual = y_test.loc[new_instances]

print("New test instances (features):")
print(X_new.to_string())
print(f"\nActual labels: {[nsp_map[v] for v in y_actual.values]}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Predict on New Instances with Each Base Model
# ──────────────────────────────────────────────────────────────
print("Predictions on New Instances:")
print("=" * 70)
print(f"{'Instance':<12} {'Actual':<14}", end="")
for name in trained_models:
    print(f"{name:<24}", end="")
print()
print("-" * 70)

for i, idx in enumerate(new_instances):
    actual = nsp_map[y_actual.loc[idx]]
    print(f"Instance {i+1:<4} {actual:<14}", end="")
    for name, model in trained_models.items():
        pred = model.predict(X_new.loc[[idx]])[0]
        pred_label = nsp_map[pred]
        status = "✅" if pred == y_actual.loc[idx] else "❌"
        print(f"{pred_label} {status}{'':>13}", end="")
    print()

---
## 3.6 Model Optimization: Hyperparameter Tuning

Use **GridSearchCV** with **Stratified K-Fold** cross-validation to find optimal hyperparameters.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Define Hyperparameter Grids
# ──────────────────────────────────────────────────────────────
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs", "liblinear"],
        "max_iter": [500, 1000, 2000],
    },
    "Random Forest": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan", "minkowski"],
    },
}

for name, grid in param_grids.items():
    total_combos = 1
    for v in grid.values():
        total_combos *= len(v)
    print(f"  {name}: {total_combos} hyperparameter combinations")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Run GridSearchCV for Each Model
# ──────────────────────────────────────────────────────────────
tuned_models = {}
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in base_models.items():
    print(f"\n{'='*60}")
    print(f"  Tuning: {name}")
    print(f"{'='*60}")

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv_strategy,
        scoring="f1_weighted",
        n_jobs=-1,
        verbose=0,
        return_train_score=True,
    )

    grid_search.fit(X_train_scaled, y_train)

    tuned_models[name] = grid_search.best_estimator_

    print(f"  Best Parameters : {grid_search.best_params_}")
    print(f"  Best CV Score   : {grid_search.best_score_:.4f} (Weighted F1)")

print(f"\n✅ Hyperparameter tuning complete for all {len(tuned_models)} models.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Evaluate Tuned Models on Test Set
# ──────────────────────────────────────────────────────────────
tuned_reports = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_scaled)

    report_dict = classification_report(
        y_test, y_pred,
        target_names=target_names,
        output_dict=True,
    )
    tuned_reports[name] = report_dict

    print("=" * 65)
    print(f"  {name} (TUNED)")
    print("=" * 65)
    print(classification_report(y_test, y_pred, target_names=target_names))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=target_names,
        yticklabels=target_names,
        linewidths=1,
        linecolor="black",
    )
    plt.title(f"Confusion Matrix — {name} (Tuned)", fontsize=14, fontweight="bold")
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("Actual", fontsize=12)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# ──────────────────────────────────────────────────────────────
# Base vs Tuned Model Comparison
# ──────────────────────────────────────────────────────────────
comparison_data = []

for name in trained_models:
    base_f1 = all_reports[name]["weighted avg"]["f1-score"]
    tuned_f1 = tuned_reports[name]["weighted avg"]["f1-score"]
    improvement = tuned_f1 - base_f1

    comparison_data.append({
        "Model": name,
        "Base Weighted F1": base_f1,
        "Tuned Weighted F1": tuned_f1,
        "Improvement": improvement,
        "Improved?": "✅ Yes" if improvement > 0 else ("➖ Same" if improvement == 0 else "❌ No"),
    })

comparison_df = pd.DataFrame(comparison_data)
print("Base vs Tuned Model Comparison:")
print("=" * 85)
print(comparison_df.to_string(index=False, float_format="{:.4f}".format))

# Find the overall best model
best_tuned_name = comparison_df.loc[comparison_df["Tuned Weighted F1"].idxmax(), "Model"]
best_tuned_score = comparison_df["Tuned Weighted F1"].max()
print(f"\n🏆 Best tuned model: {best_tuned_name} (Weighted F1 = {best_tuned_score:.4f})")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Test Tuned Models on New Data Points (Inference Demo)
# ──────────────────────────────────────────────────────────────
print("Inference with Tuned Models on New Instances:")
print("=" * 70)
print(f"{'Instance':<12} {'Actual':<14}", end="")
for name in tuned_models:
    print(f"{name:<24}", end="")
print()
print("-" * 70)

for i, idx in enumerate(new_instances):
    actual = nsp_map[y_actual.loc[idx]]
    print(f"Instance {i+1:<4} {actual:<14}", end="")
    for name, model in tuned_models.items():
        pred = model.predict(X_new.loc[[idx]])[0]
        pred_label = nsp_map[pred]
        status = "✅" if pred == y_actual.loc[idx] else "❌"
        print(f"{pred_label} {status}{'':>13}", end="")
    print()

In [ ]:
# ──────────────────────────────────────────────────────────────
# Update classification_report.json with tuned model results
# ──────────────────────────────────────────────────────────────
final_reports = {
    "base_models": all_reports,
    "tuned_models": tuned_reports,
}

with open("classification_report.json", "w") as f:
    json.dump(final_reports, f, indent=4)

print("✅ Updated classification_report.json with both base and tuned model results.")

---
## 3.7 Model Serialization

Serialize the best-performing tuned model using **joblib** for later use without retraining.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Determine the Best Model and Serialize
# ──────────────────────────────────────────────────────────────

# Map model names to file-friendly prefixes
name_to_prefix = {
    "Logistic Regression": "logreg",
    "Random Forest": "rf",
    "KNN": "knn",
}

best_model = tuned_models[best_tuned_name]
model_prefix = name_to_prefix[best_tuned_name]
model_filename = f"{model_prefix}_best_model.pkl"

# Save the model
joblib.dump(best_model, model_filename)
print(f"✅ Best model serialized: {model_filename}")
print(f"   Model: {best_tuned_name}")
print(f"   Weighted F1: {best_tuned_score:.4f}")
print(f"   File size: {os.path.getsize(model_filename) / 1024:.1f} KB")

# Also save the scaler (needed for inference on new raw data)
joblib.dump(scaler, "scaler.pkl")
print(f"\n✅ Scaler serialized: scaler.pkl")
print(f"   Feature names: {list(scaler.feature_names_in_)}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Load the Saved Model and Verify Predictions
# ──────────────────────────────────────────────────────────────
print("Loading saved model and verifying predictions...")
print("=" * 55)

# Load model from disk
loaded_model = joblib.load(model_filename)
loaded_scaler = joblib.load("scaler.pkl")

# Predict on test set using loaded model
y_pred_loaded = loaded_model.predict(X_test_scaled)

# Predict on test set using in-memory model
y_pred_original = best_model.predict(X_test_scaled)

# Verify they match
predictions_match = np.array_equal(y_pred_loaded, y_pred_original)
print(f"\nLoaded model: {model_filename}")
print(f"Predictions match original: {'✅ Yes' if predictions_match else '❌ No'}")
print(f"\nLoaded model accuracy on test set: {accuracy_score(y_test, y_pred_loaded):.4f}")

# Demonstrate inference with raw (unscaled) new data
print(f"\n{'─'*55}")
print("Inference demo with loaded model (from raw features):")
print(f"{'─'*55}")

# Take a few samples from test set (unscaled)
sample_raw = X_test.iloc[:5]
sample_scaled = loaded_scaler.transform(sample_raw)
sample_preds = loaded_model.predict(sample_scaled)

for i in range(len(sample_raw)):
    actual = nsp_map[y_test.iloc[i]]
    predicted = nsp_map[sample_preds[i]]
    status = "✅" if sample_preds[i] == y_test.iloc[i] else "❌"
    print(f"  Sample {i+1}: Actual = {actual:<12} → Predicted = {predicted:<12} {status}")

print(f"\n✅ Model serialization and verification complete!")

---
## Summary

| Step | Status |
|---|---|
| 3.1 Load Data & Import Libraries | ✅ Complete |
| 3.2 Data Preprocessing, EDA & Train-Test Split | ✅ Complete |
| 3.3 Model Training (3 Base Models) | ✅ Complete |
| 3.4 Model Evaluation (Reports + Cross-Validation) | ✅ Complete |
| 3.5 Testing on New Instances | ✅ Complete |
| 3.6 Hyperparameter Tuning (GridSearchCV) | ✅ Complete |
| 3.7 Model Serialization (joblib) | ✅ Complete |

**Output files generated:**
- `classification_report.json` — Classification reports for all base & tuned models
- `<algo>_best_model.pkl` — Best serialized model
- `scaler.pkl` — Fitted StandardScaler for inference